# LLMs4OM — Progress Update

**Two-week experiment**: evaluating new embedding retrieval models and new LLMs in the RAG-based ontology matching pipeline.

**Evaluated on**: `ncit-doid.disease` task (Bio-ML track)  
- **Source ontology**: NCIT (NCI Thesaurus) — ~5,000 disease concepts  
- **Target ontology**: DOID (Human Disease Ontology) — ~8,500 disease concepts  
- **Reference**: 4,686 ground-truth concept pairs  

**Two experiments**:
1. Fix LLM (LLaMA-3-8B), vary the **embedding retriever** → 4 new models
2. Fix retriever (BERT), vary the **LLM verifier** → 4 new models

## Work Done

### Phase 1: Onboarding & Reproducing the Original Study
- [x] Cloned repo, set up conda environment, installed all requirements
- [x] Configured HuggingFace account + token; verified RAG demo runs (Anatomy task)
- [x] Downloaded Bio-ML dataset; modified parsing to generate Bio-ML JSON directly
- [x] Ran original study models (MistralBERT, VicunaBERT, MambaBERT, LLaMA7BBert, FalconBERT) via SLURM batch on full Bio-ML track
- [x] Debugged "encoder catalog empty" issue *(fix: use catalog keys, not values, to specify models)*
- [x] Built experiment scripts for systematic runs

### Phase 2: New Model Integration & Evaluation

**Setup & Infrastructure**
- [x] Read the LLMs4OM design doc and understood the codebase architecture
- [x] Created a small test dataset (50 source concepts) to verify end-to-end pipeline quickly
- [x] Created modular test scripts: embedding retrieval test, LLM yes/no token test, RAG pipeline test
- [x] Refactored batch SLURM script to accept model name as argument (generalized pipeline)
- [x] Switched from V100 → H100 GPU for faster inference

**Experiment 1: New Embedding Retrievers (paired with LLaMA-3-8B)**
- [x] Added LLaMA-3-8B LLM class (replacing LLaMA-2)
- [x] Implemented **Qwen3-0.6B** embedding model + RAG class + tested on subset
- [x] Implemented **Qwen3-4B** embedding model + RAG class + tested on subset
- [x] Implemented **LLaMA-Nemotron-8B** embedding model + RAG class + tested on subset *(bugfix: `trust_remote_code=True`)*
- [x] Implemented **Gemma-300M** embedding model + RAG class + tested on subset
- [x] Ran all 4 models on `ncit-doid.disease` (Bio-ML track) via SLURM H100 batch

**Experiment 2: New LLM Verifiers (paired with BERT retriever)**
- [x] Implemented **Mistral-Nemo-12B** LLM + BERT RAG class *(bugfix: remove `token_type_ids` from tokenizer output)*
- [x] Implemented **Qwen2.5-7B** LLM + BERT RAG class *(bugfix: no BOS token → check token length == 1)*
- [x] Implemented **Qwen2.5-3B** LLM + BERT RAG class
- [x] Implemented **Gemma-2-9B** LLM + BERT RAG class *(requires `attn_implementation='eager'`)*
- [x] Verified yes/no token availability for each new LLM before full run
- [x] Ran all 4 models on `ncit-doid.disease` via SLURM H100 batch

## The Pipeline

```
Source ontology concepts (NCIT)
          ↓
  [Embedding Retriever]   ← Experiment 1: vary this (4 new models)
          ↓  top-k candidates per source concept
    [LLM Verifier]        ← Experiment 2: vary this (4 new models)
          ↓  yes / no per (source, candidate) pair
  Matched concept pairs
```

**Prompt sent to the LLM for each pair:**
> *Do the following two concepts refer to the same thing? Source: 'Malignant neoplasm of lung'. Target: 'lung cancer'. Answer yes or no.*

The LLM never generates free text — only the probability of the **first token** being "yes" vs "no" is used as the match confidence score.

## Data Example: Input — What a concept looks like

In [24]:
print("testing")

testing


In [5]:
ref = ds["reference"]
print(type(ref).__name__)
for split_name, split in ref.items():
    print(f"\nSplit: {split_name}, type: {type(split).__name__}, len: {len(split)}")
    first_k = next(iter(split))
    first_v = split[first_k]
    print(f"  first key: {first_k}")
    print(f"  first val type: {type(first_v).__name__}")
    print(f"  first val: {str(first_v)[:200]}")
    break


dict

Split: equiv, type: dict, len: 4
  first key: full
  first val type: list
  first val: [{'source': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C9311', 'target': 'http://purl.obolibrary.org/obo/DOID_4362'}, {'source': 'http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C8410', 't


In [11]:
import json, glob, os
from pathlib import Path
import pandas as pd

BASE = Path("/vsc-hard-mounts/leuven-data/385/vsc38504/thesis/llms4om/LLMs4OM")
OUTPUTS = BASE / "experiments/outputs/bio-ml/ncit-doid.disease"
DATASET = BASE / "datasets/bio-ml/ncit-doid.disease/om.json"
BASELINE_CSV = BASE / "experiments/results/rag-hybrid-model-results.csv"

with open(DATASET) as f:
    ds = json.load(f)

src_by_iri = {item["iri"]: item for item in ds["source"]}
tgt_by_iri = {item["iri"]: item for item in ds["target"]}
reference = {(r["source"], r["target"]) for r in ds["reference"]["equiv"]["full"]}

print(f"Source concepts (NCIT): {len(ds['source']):,}")
print(f"Target concepts (DOID): {len(ds['target']):,}")
print(f"Reference pairs:        {len(reference):,}")

# Example source concept
src = ds["source"][42]
print("\n--- Example source concept (NCIT) ---")
print(f"  Label    : {src['label']}")
print(f"  IRI      : {src['iri']}")
print(f"  Parents  : {[p['label'] for p in src.get('parents', [])[:3]]}")
print(f"  Children : {[c['label'] for c in src.get('childrens', [])[:3]]}")

# Example target concept
tgt = ds["target"][42]
print("\n--- Example target concept (DOID) ---")
print(f"  Label    : {tgt['label']}")
print(f"  IRI      : {tgt['iri']}")
print(f"  Parents  : {[p['label'] for p in tgt.get('parents', [])[:3]]}")
print(f"  Children : {[c['label'] for c in tgt.get('childrens', [])[:3]]}")

# Example reference pair
ref_list = ds["reference"]["equiv"]["full"]
pair = ref_list[0]
s = src_by_iri.get(pair["source"], {})
t = tgt_by_iri.get(pair["target"], {})
print("\n--- Example reference (ground-truth match) ---")
print(f"  Source: '{s.get('label')}' (NCIT)")
print(f"  Target: '{t.get('label')}' (DOID)")


Source concepts (NCIT): 15,762
Target concepts (DOID): 8,465
Reference pairs:        4,686

--- Example source concept (NCIT) ---
  Label    : Chronic Endometritis
  IRI      : http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C102820
  Parents  : ['Endometritis']
  Children : ['Non Specific Chronic Endometritis', 'Granulomatous Endometritis']

--- Example target concept (DOID) ---
  Label    : intestinal botulism
  IRI      : http://purl.obolibrary.org/obo/DOID_0050141
  Parents  : ['botulism']
  Children : []

--- Example reference (ground-truth match) ---
  Source: 'Malignant Cervical Neoplasm' (NCIT)
  Target: 'cervical cancer' (DOID)


## Pipeline Output Example — Retrieval + LLM Decision

In [13]:
# Load MistralNemo prediction output (label prompt)
OUTPUT_FILE = OUTPUTS / "rag-MistralNemoBertRAG-label-2026.03.19-02:56:38.json"
with open(OUTPUT_FILE) as f:
    out = json.load(f)

ir_outputs = out["generated-output"][0]["ir-outputs"]
llm_outputs = out["generated-output"][1]["llm-output"]
llm_scores = {(x["source"], x["target"]): x["score"] for x in llm_outputs}

# Find a source concept in the reference whose match appears in the top-5 candidates
ref_src_iris = {s for s, _ in reference}
example = None
for item in ir_outputs:
    src_iri = item["source"]
    if src_iri not in ref_src_iris:
        continue
    cands = item["target-cands"][:5]
    bert_sc = item["score-cands"][:5]
    ref_match = next((t for s, t in reference if s == src_iri), None)
    if ref_match and ref_match in cands:
        example = (src_iri, cands, bert_sc, ref_match)
        break

src_iri, cands, bert_sc, ref_match = example
src_label = src_by_iri.get(src_iri, {}).get("label", "?")

print(f"Source concept: '{src_label}'\n")
print(f"{'#':<3} {'Target concept (DOID)':<42} {'BERT':>6} {'LLM':>7}  Note")
print("-" * 75)

LLM_TH = 0.7
predicted = None
for i, (cand_iri, bs) in enumerate(zip(cands, bert_sc)):
    tgt_label = tgt_by_iri.get(cand_iri, {}).get("label", "?")
    ls = llm_scores.get((src_iri, cand_iri), float("nan"))
    notes = []
    if cand_iri == ref_match:
        notes.append("✓ ground truth")
    if ls >= LLM_TH and predicted is None:
        predicted = cand_iri
        notes.append("← predicted")
    print(f"{i+1:<3} {tgt_label:<42} {bs:>6.3f} {ls:>7.3f}  {', '.join(notes)}")

print(f"\nThreshold: {LLM_TH}  |  Correct: {predicted == ref_match}")

Source concept: 'Atrioventricular Septal Defect'

#   Target concept (DOID)                        BERT     LLM  Note
---------------------------------------------------------------------------
1   atrioventricular septal defect              1.000   0.988  ✓ ground truth, ← predicted
2   atrial heart septal defect                  0.913   0.987  
3   ventricular septal defect                   0.901   0.988  
4   atrial heart septal defect 5                0.875   0.988  
5   atrial heart septal defect 1                0.873   0.987  

Threshold: 0.7  |  Correct: True


## Experiment 1: New Embedding Retrievers (fixed LLM = LLaMA-3-8B)

Four new embedding models compared against the original BERT baseline.  
All paired with LLaMA-3-8B as the LLM verifier, evaluated on `ncit-doid.disease`.

1. Shows the F1 results
2. shows a sample of the output json file for a model.

In [23]:
import glob

# New embedding models (LLaMA-3 as LLM)
EMB_MODELS = {
    "LLaMA3Qwen3RAG":          "Qwen3-0.6B",
    "LLaMA3Qwen34BRAG":        "Qwen3-4B",
    "LLaMA3NemotronRAG":       "Nemotron-8B",
    "LLaMA3EmbeddingGemmaRAG": "Gemma-300M",
}

# Baseline: LLaMA-7B + BERT (from original study CSV)
df_baseline = pd.read_csv(BASELINE_CSV)
ncit_base = df_baseline[df_baseline["ontology-name"] == "ncit-doid.disease"]

rows = []

# Baseline BERT row
for enc in ["label", "label-children", "label-parent"]:
    row = ncit_base[(ncit_base["model"] == "LLaMA7BBertRAG") & (ncit_base["encoder-representation"] == enc)]
    if not row.empty:
        rows.append({"Model": "BERT (baseline)", "Retriever": "multi-qa-mpnet", "Prompt": enc, "F1": round(float(row["f1-score"].values[0]), 2)})

# New embedding model rows
for model_key, retriever_name in EMB_MODELS.items():
    files = sorted(glob.glob(str(OUTPUTS / f"rag-{model_key}-*.json")))
    for fpath in files:
        with open(fpath) as f:
            d = json.load(f)
        if "evaluation-results" not in d:
            continue
        enc = d.get("encoder-id", "?")
        f1 = round(d["evaluation-results"]["full"]["f-score"], 2)
        rows.append({"Model": model_key, "Retriever": retriever_name, "Prompt": enc, "F1": f1})

df_emb = pd.DataFrame(rows)
pivot = df_emb.pivot_table(index=["Model", "Retriever"], columns="Prompt", values="F1", aggfunc="first")
pivot = pivot[["label", "label-children", "label-parent"]]
pivot.columns = ["label F1", "label-children F1", "label-parent F1"]
pivot["Best F1"] = pivot.max(axis=1)
pivot = pivot.sort_values("Best F1", ascending=False)
print("ncit-doid.disease — Full F1 (%) by embedding model\n")
print(pivot.to_string())

ncit-doid.disease — Full F1 (%) by embedding model

                                        label F1  label-children F1  label-parent F1  Best F1
Model                   Retriever                                                            
LLaMA3Qwen34BRAG        Qwen3-4B           85.09              85.06            85.09    85.09
LLaMA3EmbeddingGemmaRAG Gemma-300M         81.90              81.87            81.90    81.90
BERT (baseline)         multi-qa-mpnet     81.68              81.55            81.68    81.68
LLaMA3Qwen3RAG          Qwen3-0.6B         81.59              81.55            81.59    81.59
LLaMA3NemotronRAG       Nemotron-8B        74.24              74.20            74.24    74.24


In [21]:
# Show what the raw output JSON looks like
sample_file = str(sorted(OUTPUTS.glob("rag-LLaMA3Qwen34BRAG-label-*.json"))[0])
with open(sample_file) as f:
    sample = json.load(f)

# Top-level structure
print("=== Top-level keys ===")
for k, v in sample.items():
    if k != "generated-output":
        print(f"  {k}: {str(v)[:80]}")

print("\n=== evaluation-results ===")
print(json.dumps(sample["evaluation-results"], indent=2))

print("\n=== generated-output structure ===")
print(f"  [0] ir-outputs  — {len(sample['generated-output'][0]['ir-outputs']):,} source concepts")
print(f"  [1] llm-output  — {len(sample['generated-output'][1]['llm-output']):,} (source, candidate) pairs")

print("\n=== Sample ir-output entry (first source concept) ===")
ir0 = sample["generated-output"][0]["ir-outputs"][0]
print(f"  source      : {ir0['source']}")
print(f"  target-cands: {ir0['target-cands'][:3]} ...")
print(f"  score-cands : {[round(s,3) for s in ir0['score-cands'][:3]]} ...")

print("\n=== Sample llm-output entry ===")
llm0 = sample["generated-output"][1]["llm-output"][0]
print(f"  source : {llm0['source']}")
print(f"  target : {llm0['target']}")
print(f"  score  : {llm0['score']:.4f}  (LLM confidence that these are the same concept)")

=== Top-level keys ===
  model: LLaMA3Qwen34BRAG
  model-path: NO MODEL LOADING IN RAG MODELS
  model-config: {'retriever-config': {'top_k': 5, 'device': 'cuda'}, 'llm-config': {'max_token_l
  dataset-info: {'track': 'bio-ml', 'ontology-name': 'ncit-doid.disease'}
  encoder-id: label
  encoder-info: PROMPT-TEMPLATE USES:LabelRAGDataset ENCODER
  response-time: 1849.861807346344
  evaluation-results: {'full': {'intersection': 3792, 'precision': 89.70901348474095, 'recall': 80.921

=== evaluation-results ===
{
  "full": {
    "intersection": 3792,
    "precision": 89.70901348474095,
    "recall": 80.92189500640204,
    "f-score": 85.08919555705148,
    "predictions-len": 4227,
    "reference-len": 4686
  },
  "test": {
    "intersection": 2658,
    "precision": 62.88147622427254,
    "recall": 81.03658536585367,
    "f-score": 70.81390702011457,
    "predictions-len": 4227,
    "reference-len": 3280
  },
  "train": {
    "intersection": 1134,
    "precision": 26.827537260468414,
    "rec

## Experiment 2: New LLM Verifiers (fixed retriever = BERT)

Four new LLMs compared against the original baselines from the study.  
All paired with BERT (`multi-qa-mpnet`) as the retriever, evaluated on `ncit-doid.disease`.

*note: result for gemma **label-children** needs to be investigated...?*

In [20]:
LLM_MODELS = {
    "MistralNemoBertRAG":  "Mistral-Nemo-12B",
    "Qwen25BertRAG":       "Qwen2.5-7B",
    "Qwen25_3BBertRAG":    "Qwen2.5-3B",
    "Gemma2_9BBertRAG":    "Gemma-2-9B",
}

# Original baselines from study CSV (BertRAG models only)
BASELINE_MODELS = [
    "LLaMA7BBertRAG", "FalconBertRAG", "VicunaBertRAG",
    "MPTBertRAG", "MambaLLMBertRAG", "MistralBertRAG",
]

rows = []

# Original baselines from CSV
for model in BASELINE_MODELS:
    for enc in ["label", "label-children", "label-parent"]:
        row = ncit_base[(ncit_base["model"] == model) & (ncit_base["encoder-representation"] == enc)]
        if not row.empty:
            rows.append({
                "Model": model,
                "LLM": model.replace("BertRAG", ""),
                "Source": "original study",
                "Prompt": enc,
                "F1": round(float(row["f1-score"].values[0]), 2)
            })

# New LLMs from output JSON files
for model_key, llm_name in LLM_MODELS.items():
    files = sorted(OUTPUTS.glob(f"rag-{model_key}-*.json"))
    for fpath in files:
        with open(fpath) as f:
            d = json.load(f)
        if "evaluation-results" not in d:
            continue
        enc = d.get("encoder-id", "?")
        f1 = round(d["evaluation-results"]["full"]["f-score"], 2)
        rows.append({
            "Model": model_key,
            "LLM": llm_name,
            "Source": "new",
            "Prompt": enc,
            "F1": f1
        })

df_llm = pd.DataFrame(rows)
pivot2 = df_llm.pivot_table(index=["Model", "LLM", "Source"], columns="Prompt", values="F1", aggfunc="first")
pivot2 = pivot2[["label", "label-children", "label-parent"]]
pivot2.columns = ["label F1", "label-children F1", "label-parent F1"]
pivot2["Best F1"] = pivot2.max(axis=1)
pivot2 = pivot2.sort_values("Best F1", ascending=False)
print("ncit-doid.disease — Full F1 (%) by LLM verifier (retriever = BERT)\n")
print(pivot2.to_string())

ncit-doid.disease — Full F1 (%) by LLM verifier (retriever = BERT)

                                                    label F1  label-children F1  label-parent F1  Best F1
Model              LLM              Source                                                               
MistralNemoBertRAG Mistral-Nemo-12B new                81.68              81.69            81.68    81.69
MPTBertRAG         MPT              original study     81.68              81.69            81.68    81.69
Qwen25_3BBertRAG   Qwen2.5-3B       new                81.68              81.64            81.68    81.68
LLaMA7BBertRAG     LLaMA7B          original study     81.68              81.55            81.68    81.68
Qwen25BertRAG      Qwen2.5-7B       new                81.68              81.66            81.68    81.68
FalconBertRAG      Falcon           original study     81.65              81.61            81.61    81.65
Gemma2_9BBertRAG   Gemma-2-9B       new                80.77              40.86     


### Next Steps

1. **Fix Gemma-2-9B** `label-children` issue — re-run that single prompt type
2. **Select best combination**: BERT retriever + best LLM → run on full Bio-ML track (all 8 tasks)
3. **Run full Bio-ML track** for all new embedding models (currently only ncit-doid)
4. **Analyse why BERT outperforms newer embedding models** — domain-specific training data?
5. **Choose top 2 embedding and LLM models to be ran as RAG**